# Tutorial 15: Transfer Learning with Static Layout Embeddings

Static v0 embeddings give every layout the same numerical language: one parameter
summary, ten geometric moments, and a signed 96 x 96 shape bitmap. This tutorial asks
whether that shared representation can reduce the number of expensive target-domain
simulations needed to learn a capacitance model.

We use only `GeneralizedCapNInterdigital` designs and treat its two simulation campaigns
as related domains:

- **X / source (`exp6`)**: 2,243 labeled layouts used to pretrain a regression head.
- **Y / target (`exp7`)**: 1,440 layouts with shifted geometry ranges.
- **A features**: compact features derived from each 9,227-dimensional v0 embedding.
- **B targets**: $C(N,S)$, $C(N,G)$, and $C(S,G)$ in fF.

The experiment compares a source-only **zero-shot** model, a **target-only** model trained
from scratch on M labels, and a **transfer** model regularized toward the source weights.
Repeated target subsampling makes label efficiency, uncertainty, and failure cases visible.

## 1. Figures of merit

No single score answers every transfer-learning question, so we report:

1. **Macro R²** across the three capacitance targets: explained target-domain variance.
2. **MAE (fF)** and **MAPE (%)**: absolute and scale-relative prediction error.
3. **Within-5% accuracy**: the percentage of held-out predictions whose relative error is
   at most 5%.
4. **Transfer gain**: transfer R² minus target-only R² at the same M.
5. **Required target labels M**: the smallest tested M whose lower 80% repeated-trial
   confidence bound reaches a requested score.

R² is the primary model-quality metric. Within-5% accuracy gives the question “D% accuracy”
a concrete operational meaning for the final M(C,D) experiment.

In [1]:
import json
import logging
import os

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from huggingface_hub import hf_hub_download
from plotly.subplots import make_subplots

from squadds.layouts import (
    StaticEmbeddingClient,
    TransferRidgeRegressor,
    V0TransferLearningStudy,
    canonical_design_id,
    compress_v0_embeddings,
    regression_scores,
    required_target_samples,
    summarize_learning_curve,
)

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 20)
logging.getLogger("httpx").setLevel(logging.WARNING)

EMBEDDING_REVISION = os.getenv("SQUADDS_EMBEDDING_REVISION", "main")
DATABASE_REVISION = os.getenv("SQUADDS_DATABASE_REVISION", "main")
EMBEDDING_REPOSITORY = "SQuADDS/SQuADDS_Layout_Embeddings"
DATABASE_REPOSITORY = "SQuADDS/SQuADDS_DB"
TARGET_NAMES = ["C(N,S)", "C(N,G)", "C(S,G)"]
PALETTE = {
    "zero-shot": "#6A4C93",
    "target-only": "#D1495B",
    "transfer": "#00798C",
}

## 2. Join simulation targets to v0 embeddings

The source JSON remains the authority for design options and simulated capacitances.
`StaticEmbeddingClient` supplies the corresponding versioned layout vectors. Stable
`design_id` values make the join reproducible.

In [2]:
embedding_client = StaticEmbeddingClient(revision=EMBEDDING_REVISION)
embeddings = embedding_client.embeddings().loc[
    lambda frame: frame["component_name"] == "GeneralizedCapNInterdigital"
]
database_path = hf_hub_download(
    repo_id=DATABASE_REPOSITORY,
    repo_type="dataset",
    filename="coupler-GeneralizedCapNInterdigital-cap_matrix.json",
    revision=DATABASE_REVISION,
)
with open(database_path) as stream:
    simulation_rows = json.load(stream)


def parse_um(value):
    return float(str(value).replace("um", ""))


records = []
for row in simulation_rows:
    options = row["design"]["design_options"]
    results = row["sim_results"]
    records.append(
        {
            "design_id": canonical_design_id("GeneralizedCapNInterdigital", options),
            "campaign": row["notes"]["source_campaign"],
            "finger_count": float(options["finger_count"]),
            "finger_length_um": parse_um(options["finger_length"]),
            "finger_width_um": parse_um(options["finger_width"]),
            "finger_gap_um": parse_um(options["finger_gap_north_south"]),
            "C(N,S)": results["north_to_south"],
            "C(N,G)": results["north_to_ground"],
            "C(S,G)": results["south_to_ground"],
        }
    )

data = embeddings.merge(pd.DataFrame(records), on="design_id", validate="one_to_one")
domain_summary = data.groupby("campaign").agg(
    layouts=("layout_id", "size"),
    finger_count=("finger_count", lambda values: f"{values.min():.0f}–{values.max():.0f}"),
    finger_length_um=("finger_length_um", lambda values: f"{values.min():.0f}–{values.max():.0f}"),
    finger_width_um=("finger_width_um", lambda values: f"{values.min():.0f}–{values.max():.0f}"),
    finger_gap_um=("finger_gap_um", lambda values: f"{values.min():.0f}–{values.max():.0f}"),
)
domain_summary

,layouts,finger_count,finger_length_um,finger_width_um,finger_gap_um
campaign,,,,,
exp6,2243,2–10,8–12,4–8,3–7
exp7,1440,2–10,5–9,3–7,2–6


In [3]:
# %% hide input
geometry_columns = [
    "finger_count",
    "finger_length_um",
    "finger_width_um",
    "finger_gap_um",
]
domain_geometry = data.melt(
    id_vars="campaign",
    value_vars=geometry_columns,
    var_name="design feature",
    value_name="value",
)
feature_labels = {
    "finger_count": "finger count",
    "finger_length_um": "finger length (um)",
    "finger_width_um": "finger width (um)",
    "finger_gap_um": "north-south gap (um)",
}
domain_geometry["design feature"] = domain_geometry["design feature"].map(feature_labels)
fig = px.box(
    domain_geometry,
    x="campaign",
    y="value",
    color="campaign",
    facet_col="design feature",
    facet_col_wrap=2,
    points=False,
    color_discrete_map={"exp6": "#D1495B", "exp7": "#00798C"},
    title="The target campaign occupies a shifted geometry regime",
)
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
fig.update_yaxes(matches=None)
fig.update_layout(height=620, showlegend=False, template="plotly_white")
fig.show()

The campaigns overlap in finger count, but `exp7` shifts the length, width, and gap
ranges downward. That is a useful transfer test: the topology is shared, while the
geometric and output distributions are not identical.

## 3. Build a source-to-target transfer study

`V0TransferLearningStudy` applies three reusable steps:

1. average-pool the 96 x 96 shape tensor to 12 x 12 while retaining all 11 non-shape
   dimensions;
2. standardize the resulting 155 features using **source-only** statistics; and
3. fit a multi-output ridge head.

The transfer head minimizes target error while penalizing movement away from the source
weights. The target-only head uses the same architecture and regularization but starts
from a zero prior, making the comparison controlled.

In [4]:
source_mask = data["campaign"] == "exp6"
target_mask = data["campaign"] == "exp7"

source_embeddings = np.vstack(data.loc[source_mask, "embedding"]).astype(np.float32)
target_embeddings = np.vstack(data.loc[target_mask, "embedding"]).astype(np.float32)
source_targets = data.loc[source_mask, TARGET_NAMES].to_numpy(dtype=float)
target_targets = data.loc[target_mask, TARGET_NAMES].to_numpy(dtype=float)

study = V0TransferLearningStudy(
    source_embeddings,
    source_targets,
    target_embeddings,
    target_targets,
    target_names=TARGET_NAMES,
    pooled_shape_size=12,
    alpha=10.0,
)

pd.Series(
    {
        "source layouts N": len(source_embeddings),
        "target layouts": len(target_embeddings),
        "raw v0 dimensions": source_embeddings.shape[1],
        "compact features A": study.source_features.shape[1],
        "mean target-to-source cosine C": study.domain_similarity()["mean"],
    },
    name="experiment",
)

source layouts N                  2243.000000
target layouts                    1440.000000
raw v0 dimensions                 9227.000000
compact features A                 155.000000
mean target-to-source cosine C       0.758188
Name: experiment, dtype: float64

In [5]:
# %% hide input
feature_blocks = pd.DataFrame(
    {
        "representation": ["raw v0", "compact transfer features"],
        "parameters + moments": [11, 11],
        "shape": [96 * 96, 12 * 12],
    }
).melt(id_vars="representation", var_name="block", value_name="dimensions")
fig = px.bar(
    feature_blocks,
    x="representation",
    y="dimensions",
    color="block",
    barmode="stack",
    log_y=True,
    text="dimensions",
    color_discrete_sequence=["#D1495B", "#00798C"],
    title="Pooling reduces fitting cost while retaining explicit shape structure",
)
fig.update_traces(textposition="inside")
fig.update_layout(height=470, template="plotly_white", yaxis_title="dimensions (log scale)")
fig.show()

In [6]:
# %% hide input
target_frame = data.loc[target_mask, ["layout_id", *geometry_columns]].reset_index(drop=True)
target_frame["cosine_to_source"] = study.target_similarity
fig = px.histogram(
    target_frame,
    x="cosine_to_source",
    nbins=38,
    marginal="box",
    color_discrete_sequence=["#00798C"],
    title="Target layouts span a broad range of similarity to the source domain",
)
fig.add_vline(
    x=study.domain_similarity()["mean"],
    line_dash="dash",
    line_color="#D1495B",
    annotation_text=f"mean C={study.domain_similarity()['mean']:.3f}",
)
fig.update_layout(height=500, template="plotly_white", yaxis_title="target layouts")
fig.show()

Here C is the cosine similarity of a target v0 embedding to the normalized **source
centroid**. It summarizes source relevance without searching for a convenient nearest
neighbor. It is useful, but it does not encode target noise, target-function complexity,
or test-set variance, so C alone cannot determine M.

## 4. Does transfer reduce the target-label requirement?

We reserve 30% of `exp7` as one fixed test set. From the remaining pool we repeatedly
sample M labels for M = 4 through 256. Every method sees the same sampled rows and test
rows on each repeat.

In [7]:
SAMPLE_SIZES = [4, 8, 16, 32, 64, 128, 256]
curves = study.learning_curve(
    SAMPLE_SIZES,
    repeats=12,
    test_fraction=0.30,
    random_seed=15,
)
r2_summary = summarize_learning_curve(curves, metric="r2", confidence=0.8).query(
    "target == 'macro'"
)
r2_summary[["method", "sample_size", "mean", "lower", "upper"]].head(9)

,method,sample_size,mean,lower,upper
3,target-only,4,0.406939,0.151859,0.602760
7,target-only,8,0.616649,0.487981,0.731019
11,target-only,16,0.800537,0.748054,0.847390
15,target-only,32,0.889275,0.863883,0.922168
19,target-only,64,0.942314,0.935860,0.947261
23,target-only,128,0.964682,0.962586,0.967008
27,target-only,256,0.974368,0.973359,0.974900
31,transfer,4,0.949580,0.941552,0.960881
35,transfer,8,0.954109,0.938518,0.969140


In [8]:
# %% hide input
fig = go.Figure()
for method in ["zero-shot", "target-only", "transfer"]:
    frame = r2_summary.loc[r2_summary["method"] == method].sort_values("sample_size")
    color = PALETTE[method]
    fig.add_trace(
        go.Scatter(
            x=np.r_[frame["sample_size"], frame["sample_size"][::-1]],
            y=np.r_[frame["upper"], frame["lower"][::-1]],
            fill="toself",
            fillcolor=color.replace("#", "rgba(") if False else color,
            opacity=0.10,
            line={"width": 0},
            hoverinfo="skip",
            showlegend=False,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=frame["sample_size"],
            y=frame["mean"],
            mode="lines+markers",
            name=method,
            line={"color": color, "width": 3},
            marker={"size": 8, "symbol": {"zero-shot": "diamond", "target-only": "circle", "transfer": "square"}[method]},
            customdata=np.column_stack([frame["lower"], frame["upper"]]),
            hovertemplate=(
                f"<b>{method}</b><br>M=%{{x}}<br>mean R²=%{{y:.3f}}"
                "<br>80% interval=%{customdata[0]:.3f}–%{customdata[1]:.3f}<extra></extra>"
            ),
        )
    )
fig.update_layout(
    title="Transfer preserves source knowledge when target labels are scarce",
    xaxis={"title": "labeled target designs M", "type": "log", "dtick": 0.30103},
    yaxis={"title": "macro R²", "range": [0.25, 1.01]},
    height=590,
    template="plotly_white",
    hovermode="x unified",
)
fig.show()

The decisive comparison is transfer versus target-only at the same M. The zero-shot line
shows what the source model already knows; transfer should eventually improve on it as
target evidence accumulates. A useful transfer method must not merely beat scratch at
M=4—it should also converge toward the full target trend instead of remaining anchored to
the source.

In [9]:
r2_requirements = pd.concat(
    [
        required_target_samples(
            curves,
            [0.90, 0.95, 0.97, 0.98],
            metric="r2",
            method=method,
            confidence=0.8,
        ).assign(model=method)
        for method in ["target-only", "transfer"]
    ],
    ignore_index=True,
)
r2_requirement_table = r2_requirements.pivot(
    index="target_score",
    columns="model",
    values="required_samples",
).rename_axis(index="required lower-bound R²")
r2_requirement_table["label saving"] = (
    r2_requirement_table["target-only"] - r2_requirement_table["transfer"]
)
r2_requirement_table

model,target-only,transfer,label saving
required lower-bound R²,,,
0.90,64,4,60
0.95,128,16,112
0.97,256,64,192
0.98,<NA>,256,<NA>


The table measures **label saving** conservatively: a model reaches a threshold only when
the lower 80% interval clears it. A missing value means the tested budget up to M=256 did
not establish that threshold reliably.

## 5. Inspect predictions with only 16 target labels

Learning curves summarize thousands of predictions. A parity plot makes the same result
concrete for one deterministic 16-label adaptation set.

In [10]:
adaptation_pool, held_out = study.target_split(test_fraction=0.30, random_seed=15)
sixteen_labels = np.random.default_rng(1516).choice(adaptation_pool, size=16, replace=False)
models = study.fit_models(sixteen_labels)

parity_records = []
for method in ["target-only", "transfer"]:
    predictions = models[method].predict(study.target_features[held_out])
    for target_index, target_name in enumerate(TARGET_NAMES):
        parity_records.extend(
            {
                "method": method,
                "target": target_name,
                "expected": expected,
                "predicted": predicted,
            }
            for expected, predicted in zip(
                study.target_targets[held_out, target_index],
                predictions[:, target_index],
            )
        )
parity = pd.DataFrame(parity_records)

In [11]:
# %% hide input
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Target-only: 16 labels", "Transfer: 16 labels"],
    horizontal_spacing=0.10,
)
for column, method in enumerate(["target-only", "transfer"], start=1):
    frame = parity.loc[(parity["method"] == method) & (parity["target"] == "C(N,S)")]
    bounds = [min(frame["expected"].min(), frame["predicted"].min()), max(frame["expected"].max(), frame["predicted"].max())]
    fig.add_trace(
        go.Scattergl(
            x=frame["expected"],
            y=frame["predicted"],
            mode="markers",
            marker={"size": 6, "opacity": 0.55, "color": PALETTE[method]},
            name=method,
            hovertemplate="simulated=%{x:.3f} fF<br>predicted=%{y:.3f} fF<extra></extra>",
        ),
        row=1,
        col=column,
    )
    fig.add_trace(
        go.Scatter(
            x=bounds,
            y=bounds,
            mode="lines",
            line={"color": "#6B7280", "dash": "dash"},
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=column,
    )
fig.update_xaxes(title_text="simulated C(N,S) (fF)")
fig.update_yaxes(title_text="predicted C(N,S) (fF)")
fig.update_layout(
    title="A source prior stabilizes the target fit in the few-label regime",
    height=520,
    template="plotly_white",
    showlegend=False,
)
fig.show()

In [12]:
# %% hide input
error_at_16 = (
    curves.loc[(curves["sample_size"] == 16) & (curves["target"] != "macro")]
    .groupby(["method", "target"], as_index=False)["mape_percent"]
    .mean()
)
fig = px.bar(
    error_at_16,
    x="target",
    y="mape_percent",
    color="method",
    barmode="group",
    color_discrete_map=PALETTE,
    title="Relative error by capacitance target at M=16",
    labels={"mape_percent": "mean absolute percentage error (%)"},
)
fig.update_layout(height=480, template="plotly_white")
fig.show()

## 6. Answering M(C,D): how many target labels are enough?

There is no universal formula using only N, C, and D. M also depends on target noise,
model capacity, how well X covers Y, the target function, the split, and the definition of
“accuracy.” The defensible answer is an empirical conditional learning curve.

For this notebook:

$$
\widehat{M}(C,D) =
\min_m \left\{
q_{0.10}\left[
\mathrm{Accuracy}_{\pm5\%}(m,C)

ight] \ge D

ight\}.
$$

- C is represented by three quantile bands of target cosine similarity to the source
  centroid.
- $\mathrm{Accuracy}_{\pm5\%}$ is the percentage of all held-out capacitance predictions
  within 5% relative error.
- $q_{0.10}$ is the lower edge of the central 80% repeated-subsampling interval.
- If no tested M reaches D, the result is “not established,” not an extrapolated promise.

In [13]:
similarity_curves = study.similarity_learning_curves(
    SAMPLE_SIZES,
    bands=3,
    repeats=12,
    test_fraction=0.30,
    random_seed=15,
)
accuracy_summary = summarize_learning_curve(
    similarity_curves,
    metric="within_5_percent",
    confidence=0.8,
).query("target == 'macro' and method == 'transfer'")

band_labels = (
    similarity_curves.groupby("similarity_band")
    .agg(
        cosine_min=("similarity_min", "first"),
        cosine_mean=("similarity_mean", "first"),
        cosine_max=("similarity_max", "first"),
    )
)
band_labels

,cosine_min,cosine_mean,cosine_max
similarity_band,,,
Q1,0.305600,0.626810,0.730339
Q2,0.730545,0.789777,0.829338
Q3,0.829443,0.857979,0.900274


In [14]:
# %% hide input
band_colors = {"Q1": "#D1495B", "Q2": "#E9C46A", "Q3": "#00798C"}
fig = go.Figure()
for band in ["Q1", "Q2", "Q3"]:
    frame = accuracy_summary.loc[accuracy_summary["similarity_band"] == band].sort_values("sample_size")
    cosine = frame["similarity_mean"].iloc[0]
    fig.add_trace(
        go.Scatter(
            x=frame["sample_size"],
            y=frame["mean"],
            mode="lines+markers",
            name=f"{band}: mean C={cosine:.3f}",
            line={"width": 3, "color": band_colors[band]},
            marker={"size": 8},
            error_y={
                "type": "data",
                "symmetric": False,
                "array": frame["upper"] - frame["mean"],
                "arrayminus": frame["mean"] - frame["lower"],
                "thickness": 1.2,
                "width": 3,
            },
            hovertemplate=(
                f"<b>{band}</b><br>M=%{{x}}<br>within-5% accuracy=%{{y:.1f}}%"
                "<extra></extra>"
            ),
        )
    )
fig.update_layout(
    title="Higher source similarity improves strict ±5% target accuracy",
    xaxis={"title": "labeled target designs M", "type": "log", "dtick": 0.30103},
    yaxis={"title": "predictions within 5% relative error (%)", "range": [20, 90]},
    height=570,
    template="plotly_white",
)
fig.show()

In [15]:
accuracy_requirements = required_target_samples(
    similarity_curves,
    [50, 60, 70, 80],
    metric="within_5_percent",
    method="transfer",
    confidence=0.8,
)
requirement_grid = accuracy_requirements.pivot(
    index="similarity_band",
    columns="target_score",
    values="required_samples",
).loc[["Q1", "Q2", "Q3"]]
requirement_grid

target_score,50.0,60.0,70.0,80.0
similarity_band,,,,
Q1,16,128,<NA>,<NA>
Q2,16,64,256,<NA>
Q3,8,16,64,256


In [16]:
# %% hide input
z = requirement_grid.apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
labels = np.full(z.shape, "not reached", dtype=object)
labels[np.isfinite(z)] = z[np.isfinite(z)].astype(int).astype(str)
fig = go.Figure(
    go.Heatmap(
        z=z,
        x=[f"D={int(value)}%" for value in requirement_grid.columns],
        y=[
            f"{band} (mean C={band_labels.loc[band, 'cosine_mean']:.3f})"
            for band in requirement_grid.index
        ],
        text=labels,
        texttemplate="%{text}",
        colorscale="Viridis_r",
        colorbar={"title": "required M"},
        hovertemplate="%{y}<br>%{x}<br>M=%{text}<extra></extra>",
        hoverongaps=False,
    )
)
fig.update_layout(
    title="Empirical M(C,D): target labels required for reliable ±5% accuracy",
    xaxis_title="requested held-out accuracy D",
    yaxis_title="target-to-source cosine band C",
    height=480,
    template="plotly_white",
)
fig.show()

### How to read the result

The heatmap is an evidence table for this dataset, model, tolerance, and split protocol.
It shows why a single scalar C cannot yield a universal M: similarity changes label
efficiency, but the answer is still conditional on the prediction task and confidence
criterion. “Not reached” means the lower confidence bound did not reach D within the
tested M ≤ 256.

For a new X→Y problem, rerun the same study with domain-appropriate targets, sample sizes,
and a tolerance that reflects engineering usefulness. Do not interpolate beyond measured
sample sizes unless you clearly label the result as a model-based extrapolation.

## 7. Reuse the APIs

The reusable workflow is intentionally short:

```python
study = V0TransferLearningStudy(
    source_embeddings,
    source_targets,
    target_embeddings,
    target_targets,
    target_names=["target 1", "target 2"],
)

curves = study.learning_curve([4, 8, 16, 32, 64], repeats=20)
summary = summarize_learning_curve(curves, metric="r2", confidence=0.8)
requirements = required_target_samples(
    curves,
    target_scores=[0.90, 0.95],
    metric="r2",
    method="transfer",
)
```

Use `study.similarity_learning_curves(...)` when the M(C,D) relationship matters.
`study.target_split(...)` and `study.fit_models(...)` expose deterministic held-out
evaluation and concrete adapted models for parity plots or downstream inference.

## 8. Conclusions

- The v0 embedding supports a transferable capacitance predictor across two shifted
  Generalized NCap campaigns.
- Transfer is most valuable in the few-label regime; target-only training catches up as M
  grows.
- R², fF error, relative error, and tolerance accuracy answer different questions and
  should be reported together.
- M is estimated from repeated target-domain learning curves. C is useful conditioning
  information, but it is not sufficient by itself.
- Confidence-aware thresholds prevent a lucky subsample from being reported as a reliable
  target-label requirement.